# Day 6 Capstone - build the TravelMind Disruption Desk

One realistic build that connects everything from today: the hand-rolled loop from Demo 1, the Strands ladder from Demo 2, and the production patterns from Demo 2b. You wire a small support desk that resolves or routes each customer message with the fewest possible model calls and never crashes in front of a customer.

- **8 TODOs.** Fill them top to bottom, then run the scoreboard for a score out of 8.
- **Scoring is offline. No AWS credentials needed.** You do need `strands-agents` installed. Building tools, agents, and graphs makes no AWS call; only running them does, and that is gated behind `RUN_LIVE`.
- Mermaid diagrams render on GitHub, nbviewer, or mermaid.live. The `show_graph()` cell prints Mermaid from your real wiring.

## What connects to what

```mermaid
flowchart TD
    D1[Demo 1: hand-rolled tool loop] --> S[Strands Agent hides that loop]
    S --> T[tool: functions the model can call]
    S --> SO[structured_output: Pydantic in, typed out]
    SO --> H[Heuristic short-circuit: skip the model]
    H --> RT[decide_route: proceed / clarify / escalate]
    RT --> GR[GraphBuilder: same routing, declarative]
    RT --> MA[Multi-agent: router plus specialists plus escalate]
    T --> RES[Resolver: tools first, model last]
    RES --> GD[guard: every failure has a home]
```

**Coverage map**

| Concept from today | Where | TODO |
|---|---|---|
| Hand-rolled loop and step guard | Part 0 | given, study it |
| `@tool` with a guard | Tools | T2 |
| Pydantic model plus heuristic | Extract | T3 |
| `decide_route` branching | Route | T4 |
| `guard` fail-safe | Guard | T5 |
| Deterministic multi-agent router | Specialists | T6 |
| `GraphBuilder` and conditional edges | Graph | T7 |
| Tools-first resolver, minimal model calls | Resolve | T8 |

## The scenario

TravelMind runs a disruption desk. Messages arrive about cancellations, refunds, rebooking, and the occasional out-of-scope rant. The desk must:

- parse the message (cheap rules first, model only when unsure),
- route it: proceed, ask one question, or hand to a human,
- for proceed, look up the booking and options with tools, and phrase a reply only if needed,
- block a refund above the desk limit until a human approves,
- survive any failure with a safe message.

```mermaid
flowchart TD
    MSG[Customer message] --> EX{Heuristic sure?}
    EX -->|yes, 0 model calls| E2[Extraction]
    EX -->|no| LX[LLM extractor] --> E2
    E2 --> R{decide_route}
    R -->|proceed| RS[resolve: lookup plus options, tools first]
    R -->|clarify| ASK[ask one question]
    R -->|escalate| HUM[human ticket]
    RS --> REF{refund over limit?}
    REF -->|yes| HUM
    REF -->|no| DONE[Resolution]
    E2 -. failure .-> HUM
```

## Three frameworks to carry through the build

**The four homes.** Every step routes to exactly one. Miss one and it becomes a production incident.

- **Proceed**: inputs are clean, do the work.
- **Recover**: something is missing but fixable, ask one question.
- **Escalate**: out of scope or over a policy limit, hand to a human, deterministically.
- **Fail safe**: an exception, degrade to a safe message and log it.

**The rule to model ladder.** Climb only when the cheaper rung cannot answer.

`rule -> deterministic tool -> single LLM call -> autonomous agent -> multi-agent`

**Minimise model calls.** Before writing `agent(...)`, ask: is this deterministic (use a rule), does it need data or an action (use a tool), or does it need language understanding or generation (then, and only then, one model call)?

$$
\text{cost per request} = \sum_{p\,\in\,\text{paths}} P(p)\cdot n_{\text{calls}}(p)\cdot c_{\text{call}}
$$

## Part 0 - where we came from (given, offline)

Demo 1 had you write the tool-use loop by hand: send the request, append the assistant `tool_use` message, run the tool, return a `tool_result` with a matching id, and loop with a step guard. Run this once and read it. Every line here is what a Strands `Agent` writes for you.

In [ ]:
# ---- given: the Demo 1 loop against an offline mock. No AWS, no Strands. ----
class _MockBedrock:
    def converse(self, modelId=None, messages=None, toolConfig=None, **kw):
        n = sum(1 for m in messages for c in m.get("content", []) if isinstance(c, dict) and "toolResult" in c)
        if toolConfig and n == 0:   # first turn: ask for the tool
            return {"output": {"message": {"role": "assistant", "content": [
                {"toolUse": {"toolUseId": "tu_1", "name": "lookup_booking", "input": {"pnr": "JX48Q2"}}}]}},
                "stopReason": "tool_use"}
        return {"output": {"message": {"role": "assistant", "content": [
            {"text": "Booking JX48Q2 (AI-302) is CANCELLED due to weather."}]}}, "stopReason": "end_turn"}

def manual_loop(user_text, max_steps=5):
    client = _MockBedrock()
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    tool_cfg = {"tools": [{"toolSpec": {"name": "lookup_booking", "description": "look up a booking",
        "inputSchema": {"json": {"type": "object", "properties": {"pnr": {"type": "string"}}, "required": ["pnr"]}}}}]}
    resp = client.converse(modelId="x", messages=messages, toolConfig=tool_cfg)
    steps = 0
    while resp["stopReason"] == "tool_use":
        if steps >= max_steps:                       # the guard Demo 1 taught
            return "step limit reached"
        steps += 1
        msg = resp["output"]["message"]; tu = msg["content"][-1]["toolUse"]
        messages.append(msg)                          # assistant turn FIRST (everyone forgets this)
        out = {"pnr": tu["input"]["pnr"], "status": "CANCELLED"}   # (would call the real tool)
        messages.append({"role": "user", "content": [
            {"toolResult": {"toolUseId": tu["toolUseId"], "content": [{"json": out}]}}]})  # id must match
        resp = client.converse(modelId="x", messages=messages, toolConfig=tool_cfg)
    return resp["output"]["message"]["content"][0]["text"]

print(manual_loop("What is the status of JX48Q2?"))
# Strands does all of the above for you. That is the entire pitch.

## Scaffolding (given, pure Python)

Domain models, the intent hints, the fake booking database, the observability log, and the graph helpers.

In [ ]:
# ---- given scaffolding. Pure Python, no AWS. ----
import re
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field, ValidationError

class Intent(str, Enum):
    BOOKING_STATUS = "booking_status"
    DISRUPTION     = "disruption"
    REBOOKING      = "rebooking"
    REFUND         = "refund"
    BAGGAGE        = "baggage"
    OTHER          = "other"

class Extraction(BaseModel):
    pnr: Optional[str] = Field(default=None)
    intent: Intent = Field(default=Intent.OTHER)
    confidence: float = Field(default=1.0, ge=0.0, le=1.0)
    ambiguity_reason: Optional[str] = Field(default=None)

class Route(str, Enum):
    WRITE    = "write"
    CLARIFY  = "clarify"
    ESCALATE = "escalate"

class Resolution(BaseModel):
    route: Route
    customer_message: str
    internal_status: str
    escalation_ticket: Optional[dict] = None
    model_calls: int = 0
    trace: list = Field(default_factory=list)

PNR_RE = re.compile(r"^[A-Z0-9]{6}$")
INTENT_HINTS = {
    Intent.DISRUPTION:     ("cancel", "cancelled", "delayed", "delay", "disrupt", "stranded"),
    Intent.REBOOKING:      ("rebook", "another flight", "next flight", "change my flight"),
    Intent.REFUND:         ("refund", "money back", "reimburse"),
    Intent.BAGGAGE:        ("baggage", "luggage", "suitcase"),
    Intent.BOOKING_STATUS: ("status", "confirmed", "is my flight"),
}
CANDIDATE = re.compile(r"\b[A-Z0-9]{6}\b")
_DB = {"JX48Q2": {"status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12", "fare": 18500, "reason": "weather"}}
TOOL_LOG = []

def find_pnr(text):
    for tok in CANDIDATE.findall(text.upper()):
        if valid_pnr(tok):
            return tok
    return None

WIRING = []
ENTRY  = {"node": None}
def add_edge_viz(builder, src, dst, route):
    builder.add_edge(src, dst, condition=route_is(route))   # the real Strands call
    WIRING.append((src, dst, route.value))
def set_entry_viz(builder, node):
    builder.set_entry_point(node)
    ENTRY["node"] = node

class _MockNode:
    def __init__(self, text): self.result = text
class MockState:
    def __init__(self, extract_json): self.results = {"extract": _MockNode(extract_json)}
def _extraction_from_state(state):
    node = state.results.get("extract")
    if not node:
        return None
    try:
        return Extraction.model_validate_json(str(node.result))
    except Exception:
        return None

def show_graph():
    print("entry:", ENTRY["node"])
    for s, d, r in WIRING:
        print("  " + s + " --[" + r + "]--> " + d)
    print("\nmermaid (paste into a Mermaid viewer):\n")
    lines = ["flowchart TD",
             '    extract["extract"]', '    write["write"]', '    clarify["clarify"]', '    escalate["escalate"]']
    for s, d, r in WIRING:
        lines.append("    " + s + " -->|" + r + "| " + d)
    print("\n".join(lines))

def dry_run(sample):
    state = MockState(sample.model_dump_json())
    fired = [x for x in (Route.WRITE, Route.CLARIFY, Route.ESCALATE) if route_is(x)(state)]
    b = fired[0].value if len(fired) == 1 else ("NONE/MULTI " + str([f.value for f in fired]))
    print("  extract --> " + b + "   (intent=" + sample.intent.value + ", pnr=" + str(sample.pnr) + ")")

SAMPLES = [
    Extraction(pnr="JX48Q2", intent=Intent.DISRUPTION,     confidence=0.95),
    Extraction(pnr=None,     intent=Intent.REBOOKING,      confidence=0.90),
    Extraction(pnr="JX48Q2", intent=Intent.OTHER,          confidence=0.90),
]
print("scaffolding ready")

## Strands setup (given)

Imports, the shared model, and the four graph node agents. Everything here constructs offline.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from strands.agent.conversation_manager import SlidingWindowConversationManager

REGION = "us-west-2"
MODEL  = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
bedrock = BedrockModel(model_id=MODEL, region_name=REGION, temperature=0.2)   # constructs offline

n_extract  = Agent(model=bedrock, callback_handler=None, system_prompt="Return ONLY JSON for the extraction.")
n_write    = Agent(model=bedrock, system_prompt="Write a warm 2-sentence reply from the JSON details.")
n_clarify  = Agent(model=bedrock, system_prompt="Ask ONE specific question to recover the missing detail.")
n_escalate = Agent(model=bedrock, system_prompt="Say the issue is going to a human specialist, in 2 sentences.")
print("strands setup ready")

## TODO 1 - `valid_pnr`

A record locator is 6 characters and mixes letters with digits. Drop the mixed rule and `FLIGHT` sails through as a booking reference.

| input | valid |
|---|---|
| `JX48Q2` | yes |
| `FLIGHT` | no (no digit) |
| `123456` | no (no letter) |
| `JX4Q` | no (wrong length) |

In [ ]:
def valid_pnr(pnr):
    if not pnr:
        return False
    p = pnr.strip().upper()
    # TODO 1: require a PNR_RE match AND at least one digit AND at least one letter.
    #   hint: bool(PNR_RE.match(p)) and any(c.isdigit() for c in p) and any(c.isalpha() for c in p)
    return bool(PNR_RE.match(p))   # <-- extend this

print(valid_pnr("JX48Q2"), valid_pnr("FLIGHT"), valid_pnr("123456"))   # want: True False False

## TODO 2 - `lookup_booking` as a `@tool`

**Trick from Demo 2b: tools return structured errors, they do not raise.** A tool that throws kills the run or forces the model to guess. A tool that returns `found=false` lets the caller decide calmly.

Two guards: reject a malformed PNR before touching the backend, and return `found=false` for an unknown one.

A `@tool` function is still a normal function. Calling `lookup_booking("JX48Q2")` directly returns your dict, so you can test and reuse it without any model.

In [ ]:
@tool
def lookup_booking(pnr: str) -> dict:
    """Look up a booking by its 6-character PNR. Returns found=false for bad or unknown PNRs."""
    TOOL_LOG.append(("lookup_booking", pnr))
    p = (pnr or "").strip().upper()
    # TODO 2: complete the two guards, then the success return.
    #   1) if not valid_pnr(p): return {"found": False, "error": "invalid_pnr_format", "pnr": p}
    #   2) rec = _DB.get(p);  if not rec: return {"found": False, "error": "not_found", "pnr": p}
    #   3) return {"found": True, "pnr": p, **rec}
    return {"found": False, "error": "not_implemented", "pnr": p}   # <-- replace with 1, 2, 3

print(lookup_booking("JX48Q2"))   # want: found True, flight AI-302
print(lookup_booking("FLIGHT"))   # want: found False (invalid format)

## More tools and the specialists (given)

- `get_rebooking_options`: alternative flights.
- `issue_refund`: **policy gate baked into the tool.** Anything over `REFUND_LIMIT` returns `needs_approval` instead of paying out. The SDK-native way to gate a call is a `BeforeToolCallEvent` hook; an in-tool guard is the simplest version and always runs.
- Two specialist agents wrapped as tools, plus an escalate tool and an orchestrator (the agents-as-tools pattern from Demo 2b). Constructed offline.

In [ ]:
REFUND_LIMIT = 15000

@tool
def get_rebooking_options(pnr: str) -> list:
    """Alternative flights for a disrupted booking. Call after a successful lookup_booking."""
    TOOL_LOG.append(("get_rebooking_options", pnr))
    return [{"flight": "AI-318", "dep": "18:40"}, {"flight": "6E-552", "dep": "21:15"}]

@tool
def issue_refund(pnr: str, amount: int) -> dict:
    """Issue a refund. Amounts over the desk limit return needs_approval for a human."""
    TOOL_LOG.append(("issue_refund", pnr, amount))
    if amount > REFUND_LIMIT:
        return {"status": "needs_approval", "pnr": pnr, "amount": amount, "limit": REFUND_LIMIT}
    return {"status": "refunded", "pnr": pnr, "amount": amount}

billing_agent = Agent(model=bedrock, callback_handler=None,
    system_prompt="Billing specialist. Precise about refunds and fees. 2 to 3 sentences.")
disruption_agent = Agent(model=bedrock, callback_handler=None,
    tools=[lookup_booking, get_rebooking_options],
    system_prompt="Flight-disruption specialist. Look up the booking before answering. Never invent details.")

@tool
def handle_billing(question: str) -> str:
    """Refunds, fees, and charges."""
    return str(billing_agent(question))

@tool
def handle_disruption(question: str) -> str:
    """Delays, cancellations, rebooking."""
    return str(disruption_agent(question))

@tool
def escalate_to_human(question: str) -> str:
    """Out-of-scope or non-standard requests, routed to a human queue."""
    return "Raised to a human specialist; the customer will be contacted shortly."

orchestrator = Agent(model=bedrock,
    tools=[handle_billing, handle_disruption, escalate_to_human],
    system_prompt="Route to the right specialist tool, then reply under 60 words. Escalate anything out of scope.")
print("tools and specialists ready")

## TODO 3 - `cheap_extract` (skip the model)

Two signals: a PNR from the regex, an intent from the keyword table. Trust the shortcut only when both are unambiguous, meaning exactly one intent hit and a valid PNR. Any doubt returns `None`, and the LLM extractor (given, gated) takes over. A heuristic that guesses on ambiguous input is worse than none, because it fails quietly.

In [ ]:
def cheap_extract(msg):
    pnr  = find_pnr(msg)
    hits = [i for i, words in INTENT_HINTS.items() if any(w in msg.lower() for w in words)]
    # TODO 3: return an Extraction only when there is exactly one intent hit AND a valid pnr; else None.
    if False:   # <-- replace this condition
        return Extraction(pnr=pnr, intent=hits[0], confidence=0.95)
    return None

print(cheap_extract("My cancelled flight, PNR JX48Q2"))          # want: an Extraction
print(cheap_extract("My flight is a mess, help"))                # want: None

## TODO 4 - `decide_route` (the branch)

Order encodes precedence. Check `OTHER` first: an out-of-scope request with a valid PNR must still reach a human, not the writer.

**Qualifying matrix**

| intent standard? | valid PNR? | ambiguous or low confidence? | route |
|---|---|---|---|
| no | any | any | ESCALATE |
| yes | no | any | CLARIFY |
| yes | yes | yes | CLARIFY |
| yes | yes | no | WRITE |

In [ ]:
def decide_route(ex: Extraction) -> Route:
    # TODO 4: apply the matrix IN ORDER.
    #   1) ex.intent == Intent.OTHER                        -> Route.ESCALATE
    #   2) not valid_pnr(ex.pnr)                             -> Route.CLARIFY
    #   3) ex.ambiguity_reason or ex.confidence < 0.6        -> Route.CLARIFY
    #   4) otherwise                                         -> Route.WRITE
    return Route.WRITE   # <-- replace with the four rules

mk = lambda **k: Extraction(**{"pnr": "JX48Q2", "intent": Intent.DISRUPTION, "confidence": 0.95, **k})
print(decide_route(mk(intent=Intent.OTHER)), decide_route(mk(pnr=None)), decide_route(mk()))
# want: Route.ESCALATE Route.CLARIFY Route.WRITE

## TODO 5 - `guard` (fail safe)

A uniform envelope `(ok, value)` plus a recorded reason. Callers branch on `ok` and never handle exception types at the call site. In production the errors you expect are `EventLoopException`, `ContextWindowOverflowException`, `MaxTokensReachedException`, and pydantic `ValidationError`. Throttling is retried internally with backoff, so it rarely reaches you.

In [ ]:
def guard(label, fn, trace):
    # TODO 5: fill both branches.
    #   success -> trace.append((label, "ok"));                          return True, value
    #   failure -> trace.append((label, "error:" + type(err).__name__)); return False, None
    try:
        pass   # <-- run fn(), record ok, return (True, value)
    except Exception as err:
        pass   # <-- record the error, return (False, None)

tr = []
print(guard("a", lambda: 42, tr), tr)   # want: (True, 42) [('a', 'ok')]

## TODO 6 - `pick_specialist` (cheap multi-agent routing)

Demo 2b gave the orchestrator every query. Cheaper: a keyword router sends clear cases straight to a specialist and wakes the orchestrator model only for genuinely mixed messages. Return `"billing"`, `"disruption"`, or `None` (let the orchestrator decide).

In [ ]:
def pick_specialist(msg: str):
    m = msg.lower()
    bill = any(w in m for w in ("refund", "fee", "charge", "money"))
    disr = any(w in m for w in ("cancel", "delay", "rebook", "disrupt", "stranded"))
    # TODO 6: only-billing -> "billing"; only-disruption -> "disruption"; mixed or neither -> None
    return None   # <-- replace

print(pick_specialist("I want a refund"), pick_specialist("my flight was cancelled"),
      pick_specialist("refund for my cancelled flight"))   # want: billing disruption None

## TODO 7 - `route_is` and the graph

Two fills. First the condition: a Strands edge condition is a `state -> bool`. Reuse `decide_route`; on junk (`ex is None`) only the escalate edge fires, so a broken extractor still lands a human. Then wire three conditional edges from `extract` and set the entry point.

`GraphBuilder`: `add_node`, `add_edge(condition=...)`, `set_entry_point`, `build`. Python fires a node when any incoming edge is satisfied, and your conditions are mutually exclusive, so exactly one branch runs.

`build()` prints a harmless warning about execution limits. Our graph is acyclic, so it cannot loop. **Trick:** for graphs with cycles, `builder.set_max_node_executions(n)` bounds them, the graph-level version of the step guard from Part 0.

In [ ]:
def route_is(target: Route):
    def _cond(state):
        ex = _extraction_from_state(state)
        # TODO 7a: on junk only ESCALATE fires; otherwise fire when decide_route(ex) == target
        if ex is None:
            return False   # <-- fix
        return False       # <-- fix
    return _cond

builder = GraphBuilder()
builder.add_node(n_extract,  "extract")
builder.add_node(n_write,    "write")
builder.add_node(n_clarify,  "clarify")
builder.add_node(n_escalate, "escalate")

# TODO 7b: wire three conditional edges from "extract", then set the entry point.
#   add_edge_viz(builder, "extract", "write",    Route.____)
#   add_edge_viz(builder, "extract", "clarify",  Route.____)
#   add_edge_viz(builder, "extract", "escalate", Route.____)
#   set_entry_viz(builder, "____")


try:
    graph = builder.build()
    print("graph built:", graph is not None, "| edges:", len(WIRING), "| entry:", ENTRY["node"])
except Exception as e:
    graph = None
    print("build failed, finish TODO 7b ->", type(e).__name__)

## TODO 8 - `resolve` (the capstone: tools first, model last)

Given an `Extraction`, produce a `Resolution`. Escalate and clarify are handed to you. Your job is the proceed path: call tools deterministically, guard each call, and keep `model_calls` at zero. Phrasing a nicer reply is the only place a model would earn a call, and it is optional.

```mermaid
flowchart LR
    A[Extraction, proceed] --> B[lookup_booking]
    B -->|found false| C[clarify]
    B -->|found| D[get_rebooking_options]
    D --> E[Resolution, model_calls 0]
    E -. optional .-> F[one LLM call to phrase]
```

In [ ]:
def resolve(ex: Extraction) -> Resolution:
    TOOL_LOG.clear()
    trace = []
    route = decide_route(ex)

    if route == Route.ESCALATE:                                    # given
        ticket = {"queue": "tier2_human", "reason": "non_standard_intent",
                  "pnr": ex.pnr, "intent": ex.intent.value}
        return Resolution(route=route, customer_message="Your request is going to a human specialist.",
                          internal_status="escalated", escalation_ticket=ticket, model_calls=0, trace=trace)

    if route == Route.CLARIFY:                                     # given
        return Resolution(route=route, customer_message="Could you share your 6-character PNR so I can pull up the booking?",
                          internal_status="awaiting_customer", model_calls=0, trace=trace)

    # PROCEED. TODO 8: tools first, keep model_calls at 0.
    #   1) ok, booking = guard("lookup", lambda: lookup_booking(ex.pnr), trace)
    #   2) if (not ok) or (not booking.get("found")):
    #          return Resolution(route=Route.CLARIFY, customer_message="I could not find that PNR, please recheck it.",
    #                            internal_status="awaiting_customer", model_calls=0, trace=trace)
    #   3) ok, options = guard("options", lambda: get_rebooking_options(ex.pnr), trace)
    #   4) msg = f"Booking {booking['pnr']} on {booking['flight']} is {booking['status']}. Rebooking options are ready."
    #      return Resolution(route=route, customer_message=msg, internal_status="answered", model_calls=0, trace=trace)
    return Resolution(route=route, customer_message="TODO", internal_status="todo", model_calls=0, trace=trace)  # <-- replace

print(resolve(Extraction(pnr="JX48Q2", intent=Intent.DISRUPTION, confidence=0.95)).model_dump())

## See the graph, run the desk offline

`show_graph()` prints your topology and Mermaid. `dry_run()` routes sample extractions through your edges. Then run `resolve` on real messages, all with zero model calls.

In [ ]:
show_graph()
print("\n--- dry run (routing) ---")
for s in SAMPLES:
    dry_run(s)

print("\n--- resolve (full desk, offline) ---")
for ex in [
    Extraction(pnr="JX48Q2", intent=Intent.DISRUPTION, confidence=0.95),   # proceed
    Extraction(pnr=None,     intent=Intent.REBOOKING,  confidence=0.90),    # clarify
    Extraction(pnr="JX48Q2", intent=Intent.OTHER,      confidence=0.90),    # escalate
]:
    r = resolve(ex)
    print(f"  route={r.route.value:8}  model_calls={r.model_calls}  status={r.internal_status}")
    print("     msg:", r.customer_message)
    print("     trace:", r.trace)

## Optional: run it for real

Needs AWS credentials and Sonnet 4.5 access in `us-west-2`. Leave `RUN_LIVE = False` to skip. This shows the graph routing live and the orchestrator picking a specialist.

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    graph("Is my flight confirmed? PNR JX48Q2")                 # graph routes; one branch replies
    print(orchestrator("I want a refund for the fees on my cancelled flight JX48Q2"))
else:
    print("RUN_LIVE is False. Offline dry-run and resolve above already exercised your logic.")

## Decision matrices to keep

**Which orchestration pattern?**

| Pattern | Control | Deterministic | Observability | Model spend | Reach for it when |
|---|---|---|---|---|---|
| Heuristic + plain Python | total | yes | your logs | lowest | rules cover most traffic |
| Agents as tools | high | mostly | per-tool logs | medium | one lead delegates to specialists |
| Graph | high | yes | built-in trace | medium | approvals, gates, branches, parallel steps |
| Swarm | low | no | debug logs | highest | the sequence cannot be known upfront |

**Rule, tool, or model?**

| Mechanism | Use when | Cost | Main risk |
|---|---|---|---|
| Rule | the answer is deterministic | near zero | brittle if inputs drift |
| Deterministic tool | you need data or a side effect | cheap | integration failure |
| LLM call | fuzzy language in or out | highest | latency, spend, hallucination |

## Scoreboard

Run last. Asserts all 8 TODOs and prints a score out of 8.

In [ ]:
score = 0; total = 0
def check(label, test):
    global score, total
    total += 1
    try:
        test(); score += 1; print("  PASS  " + label)
    except Exception as e:
        print("  FAIL  " + label + "  ->  " + (str(e) or type(e).__name__))
def _raise(): raise RuntimeError("boom")
def _mk(**k): return Extraction(**{"pnr": "JX48Q2", "intent": Intent.DISRUPTION, "confidence": 0.95, **k})

def t1():
    assert valid_pnr("JX48Q2") and valid_pnr("ABC123")
    assert not valid_pnr("FLIGHT") and not valid_pnr("123456") and not valid_pnr("JX4Q") and not valid_pnr(None)

def t2():
    r = lookup_booking("JX48Q2")
    assert r["found"] and r["flight"] == "AI-302"
    assert lookup_booking("FLIGHT")["found"] is False       # invalid format guard
    assert lookup_booking("ZZ99ZZ")["found"] is False       # valid format, not in DB

def t3():
    e = cheap_extract("My cancelled flight, PNR JX48Q2")
    assert e is not None and e.pnr == "JX48Q2" and e.intent == Intent.DISRUPTION
    assert cheap_extract("My flight is a mess, help") is None
    assert cheap_extract("refund and cancelled, PNR JX48Q2") is None    # two intents -> None

def t4():
    assert decide_route(_mk(intent=Intent.OTHER)) == Route.ESCALATE
    assert decide_route(_mk()) == Route.WRITE
    assert decide_route(_mk(pnr=None)) == Route.CLARIFY
    assert decide_route(_mk(ambiguity_reason="two bookings")) == Route.CLARIFY
    assert decide_route(_mk(confidence=0.3)) == Route.CLARIFY

def t5():
    tr = []
    ok, v = guard("a", lambda: 42, tr)
    assert ok and v == 42 and tr[-1] == ("a", "ok")
    ok, v = guard("b", _raise, tr)
    assert (not ok) and v is None and "error" in tr[-1][1]

def t6():
    assert pick_specialist("I want a refund") == "billing"
    assert pick_specialist("my flight was cancelled") == "disruption"
    assert pick_specialist("refund for my cancelled flight") is None

def t7():
    routes = {(s, d): r for s, d, r in WIRING}
    assert routes.get(("extract", "write")) == "write"
    assert routes.get(("extract", "clarify")) == "clarify"
    assert routes.get(("extract", "escalate")) == "escalate"
    assert ENTRY["node"] == "extract" and graph is not None
    sW = MockState(_mk().model_dump_json())
    assert route_is(Route.WRITE)(sW) and not route_is(Route.ESCALATE)(sW)
    assert route_is(Route.ESCALATE)(MockState("nope"))

def t8():
    r = resolve(_mk())
    assert r.route == Route.WRITE and r.internal_status == "answered" and r.model_calls == 0
    labels = [t[0] for t in r.trace]
    assert "lookup" in labels and "options" in labels
    tools = [c[0] for c in TOOL_LOG]
    assert "lookup_booking" in tools and "get_rebooking_options" in tools
    assert resolve(_mk(intent=Intent.OTHER)).escalation_ticket is not None
    assert resolve(_mk(pnr=None)).route == Route.CLARIFY
    assert resolve(_mk(pnr="ZZ99ZZ")).route == Route.CLARIFY   # valid format, not found -> clarify

for lbl, fn in [("T1  valid_pnr", t1), ("T2  lookup_booking tool", t2), ("T3  cheap_extract", t3),
                ("T4  decide_route", t4), ("T5  guard", t5), ("T6  pick_specialist", t6),
                ("T7  route_is + graph wiring", t7), ("T8  resolve tools-first", t8)]:
    check(lbl, fn)

print("\nSCORE: " + str(score) + " / " + str(total))
print("Desk is production-shaped. Nice work." if score == total else "Fix the FAILs above, then re-run.")

## Recap

- Demo 1 taught the loop; Strands writes it. You spent the saved effort on routing, guards, and tools.
- Cheap first: a rule beat a model call on the common path, and a keyword router beat the orchestrator.
- Every message got a home: proceed, clarify, escalate, or fail safe. Escalation and the refund gate are deterministic and auditable.
- The resolver ran the whole happy path with zero model calls; the model is optional polish.

Skeptic's exit question: you just built a desk that mostly does not call a model. What did the model actually buy you here, and where is the one place it is genuinely worth the latency and the risk?

Next, **Demo 3** hosts a hardened agent on **AgentCore**.